In [11]:
#imports
import chromadb
from chromadb.utils import embedding_functions
import json
from pprint import pprint
from pypdf import PdfReader

In [12]:
# initialize chroma client
chroma_client = chromadb.PersistentClient(path="chromadb_data/")

In [13]:
# load career data
import json

with open("data/careers.json", "r", encoding="utf-8") as f:
    careers = json.load(f)
print(f"Loaded {len(careers)} career entries.")

Loaded 16 career entries.


In [14]:
# prepare for chroma ingestion
documents = []
metadatas = []
ids = []

for c in careers:
    text_block = f"""
Title: {c['title']}
Category: {c['category']}
Seniority: {c['seniority']}

Summary: {c['summary']}

Ideal background: {c['ideal_background']}

Core skills: {', '.join(c['core_skills'])}
Nice to have skills: {', '.join(c['nice_to_have_skills'])}

Interests fit: {', '.join(c['interests_fit'])}
Personality fit: {', '.join(c['personality_fit'])}

Typical tasks: {', '.join(c['typical_tasks'])}

Common job titles: {', '.join(c['common_job_titles'])}

Suggested learning paths: {', '.join(c['suggested_learning_paths'])}

Tags: {', '.join(c['tags'])}
"""
    documents.append(text_block)
    metadatas.append({
        "id": c["id"],
        "title": c["title"],
        "category": c["category"],
        "seniority": c["seniority"]
    })
    ids.append(c["id"])


In [15]:
#All three lists must have the same length
#Every document has its metadata and unique id
len(documents) == len(ids) == len(metadatas)

True

In [16]:
collections = chroma_client.get_or_create_collection(name="careers")

In [17]:
collection=collections.upsert(documents=documents, ids=ids,metadatas=metadatas)

In [18]:
def read_pdf(path: str) -> str:
    """
    Read PDF file and return its full text
    """
    reader = PdfReader(path)
    text = ""
    
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text 

    return text

In [20]:
# querying the collection
from pprint import pprint

resume_path="data/resumes/samaShalabiCV(AI).pdf"
profile_text = read_pdf(resume_path)

results = collections.query(
    query_texts=[profile_text],  # use the existing Collection instance `collections`
    n_results=5,
    # where_document={'$contains': 'Scientist'}
)
pprint(results)


{'data': None,
 'distances': [[0.5026835203170776,
                0.5355764627456665,
                0.5372484922409058,
                0.5767070055007935,
                0.595772922039032]],
 'documents': [['\n'
                'Title: AI Engineer\n'
                'Category: AI / Automation\n'
                'Seniority: all-levels\n'
                '\n'
                'Summary: AI engineers build AI systems, pipelines, and '
                'integrate LLMs.\n'
                '\n'
                'Ideal background: ML foundations, software engineering.\n'
                '\n'
                'Core skills: Python, ML, LLM APIs, Vector DBs\n'
                'Nice to have skills: Prompt Engineering\n'
                '\n'
                'Interests fit: AI, Automation\n'
                'Personality fit: Innovative\n'
                '\n'
                'Typical tasks: Implement AI features, Optimize models\n'
                '\n'
                'Common job titles: AI Enginee

In [37]:
# functions to retrieve careers based on profile text
# def get_careers_collection():
#     chroma_client = chromadb.PersistentClient(path="chromadb_data/")

#     embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
#         model_name="all-MiniLM-L6-v2"
#     )

#     collection = chroma_client.get_or_create_collection(
#         name="careers",
#         embedding_function=embedding_fn
#     )

#     return collection

def retrieve_careers(profile_text: str, k: int = 5):
    """
    Takes a text describing the candidate (resume text or summary)
    and returns top-k relevant careers from Chroma.
    """

    results = collections.query(
        query_texts=[profile_text],
        n_results=k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    ids = results["ids"][0]

    career_results = []
    for doc, meta, _id in zip(docs, metas, ids):
        career_results.append({
            "id": _id,
            "title": meta.get("title"),
            "category": meta.get("category"),
            "seniority": meta.get("seniority"),
            "raw_text": doc
        })

    return career_results


In [39]:
profile_text = """
Computer science student with strong skills in Python, machine learning, and data visualization.
Experience in building ML models, working with pandas and scikit-learn, and creating dashboards.
Interested in AI, data science, and solving analytical problems.
"""

results = retrieve_careers(profile_text, k=5)

pprint(results)

[{'category': 'Technology / Data',
  'id': 'data_scientist_entry',
  'raw_text': '\n'
              'Title: Data Scientist\n'
              'Category: Technology / Data\n'
              'Seniority: entry-level\n'
              '\n'
              'Summary: Data scientists use programming, statistics, and '
              'domain knowledge to extract insights from data and build '
              'predictive models that help businesses make decisions.\n'
              '\n'
              'Ideal background: Good for people who enjoy math, logical '
              'thinking, coding, and working with large datasets. Often fits '
              'CS, engineering, math, or statistics students.\n'
              '\n'
              'Core skills: Python, Statistics, Machine Learning, SQL, Data '
              'Visualization\n'
              'Nice to have skills: Deep Learning, Big Data Tools, Cloud '
              'Platforms\n'
              '\n'
              'Interests fit: Enjoys working with numbers